# Submissão A – Implementação com numpy  
**Grupo 7 | MIA | Aprendizagem Profunda**  
**Modelos:** Regressão Logística (baseline) + Deep Neural Network  
**Features:** TF-IDF implementado do zero + features estilísticas  
> ⚠️ Este notebook usa **apenas numpy e pandas** (sem bibliotecas de ML).


In [1]:
import numpy as np
import pandas as pd
import re
import pickle
from collections import Counter
import os

np.random.seed(42)
print('Dependências carregadas. numpy:', np.__version__)


Dependências carregadas. numpy: 2.4.2


## 1. Configuração e Constantes


In [2]:
# Classes possíveis (por ordem alfabética para consistência)
CLASSES    = ['Anthropic', 'Google', 'Human', 'Meta', 'OpenAI']
LABEL2IDX  = {c: i for i, c in enumerate(CLASSES)}
IDX2LABEL  = {i: c for i, c in enumerate(CLASSES)}

# Paths – ajustar conforme a localização dos ficheiros
TRAIN_PATH = 'dataset-exemplos.csv' # ou 'training_dataset.csv'
TEST_PATH  = 'subm1.csv'
OUT_PATH   = 'subm1-g7-MIA-A.csv'
MODEL_PATH = 'model_A.pkl'

print('Classes:', CLASSES)


Classes: ['Anthropic', 'Google', 'Human', 'Meta', 'OpenAI']


## 2. Carregamento dos Dados


In [3]:
train_df = pd.read_csv(TRAIN_PATH, sep=';')
test_df  = pd.read_csv(TEST_PATH,  sep=';', encoding='utf-8-sig')

print(f'Dados de treino: {len(train_df)} exemplos')
print(f'Dados a classificar: {len(test_df)} exemplos')
print('\nDistribuição das classes (treino):')
print(train_df['Label'].value_counts())
print('\nPrimeiros exemplos de treino:')
train_df.head(3)


Dados de treino: 125 exemplos
Dados a classificar: 150 exemplos

Distribuição das classes (treino):
Label
Human        52
Anthropic    23
Meta         17
OpenAI       17
Google       16
Name: count, dtype: int64

Primeiros exemplos de treino:


,ID,Text,Label
0,D1-1,"It is an approximation useful in chemistry, bu...",Human
1,D1-2,"PET scanning, or Positron Emission Tomography,...",Meta
2,D1-3,Positron Emission Tomography (PET) scanning is...,Google


## 3. Pré-processamento e Feature Engineering


In [4]:
# Stop words em inglês
STOPWORDS = set([
    'the','a','an','and','or','but','in','on','at','to','for','of','with',
    'by','from','is','are','was','were','be','been','being','have','has',
    'had','do','does','did','will','would','could','should','may','might',
    'shall','can','this','that','these','those','it','its','they','them',
    'their','we','our','you','your','he','she','his','her','i','my','me',
    'as','not','so','if','up','out','about','into','than','more','also',
    'which','when','where','who','how','what','such','other','each','both',
    'all','some','any','only','after','before','between','through','over',
])

def tokenize(text):
    """Tokenização básica: lowercase, remoção de pontuação, remoção de stop words."""
    text = text.lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    tokens = text.split()
    return [t for t in tokens if t not in STOPWORDS and len(t) > 2]

def extract_stylistic_features(text):
    """
    Extrai features estilísticas independentes do conteúdo temático.
    Estas features procuram capturar padrões de escrita específicos
    de cada origem (humano vs. diferentes LLMs).
    """
    words = text.split()
    sentences = re.split(r'[.!?]+', text.strip())
    sentences = [s.strip() for s in sentences if s.strip()]
    n_words    = max(len(words), 1)
    n_sents    = max(len(sentences), 1)

    avg_word_len  = np.mean([len(w) for w in words]) if words else 0
    avg_sent_len  = n_words / n_sents
    vocab_richness = len(set(words)) / n_words

    n_commas   = text.count(',')
    n_semicol  = text.count(';')
    n_colons   = text.count(':')
    n_parens   = text.count('(') + text.count(')')
    n_dash     = text.count('-') + text.count('–') + text.count('—')

    func_words = {'is','are','the','a','an','this','that','these','those','which','where','when'}
    func_ratio = sum(1 for w in words if w.lower() in func_words) / n_words
    cap_ratio  = sum(1 for w in words[1:] if w and w[0].isupper()) / n_words

    first_word = words[0].lower() if words else ''

    return np.array([
        avg_word_len, avg_sent_len, vocab_richness,
        n_commas / n_words, n_semicol / n_words,
        n_colons / n_words, n_parens / n_words, n_dash / n_words,
        func_ratio, cap_ratio,
        float(first_word == 'the'), float(first_word in ['a', 'an']),
        n_words / 100.0,
    ], dtype=np.float32)

print('Funções de pré-processamento definidas.')


Funções de pré-processamento definidas.


In [5]:
class TFIDFVectorizer:
    """
    TF-IDF implementado do zero com numpy.
    TF  = frequência relativa do termo no documento
    IDF = log((N+1)/(df+1)) + 1  (suavizado)
    Normalização final: L2 por documento.
    """
    def __init__(self, max_features=2000):
        self.max_features = max_features
        self.vocab  = {}
        self.idf_   = None

    def fit(self, texts):
        N  = len(texts)
        tokenized = [tokenize(t) for t in texts]
        df = Counter()
        for tokens in tokenized:
            df.update(set(tokens))  # document frequency
        # Manter as max_features palavras com maior df
        top = [w for w, _ in df.most_common(self.max_features)]
        self.vocab = {w: i for i, w in enumerate(top)}
        V = len(self.vocab)
        self.idf_ = np.array(
            [np.log((N + 1) / (df[w] + 1)) + 1 for w in top],
            dtype=np.float32
        )
        return self

    def transform(self, texts):
        V, N = len(self.vocab), len(texts)
        X = np.zeros((N, V), dtype=np.float32)
        for i, text in enumerate(texts):
            tokens = tokenize(text)
            tf = Counter(tokens)
            n_tok = max(len(tokens), 1)
            for w, cnt in tf.items():
                if w in self.vocab:
                    j = self.vocab[w]
                    X[i, j] = (cnt / n_tok) * self.idf_[j]
            norm = np.linalg.norm(X[i])
            if norm > 0:
                X[i] /= norm
        return X

    def fit_transform(self, texts):
        return self.fit(texts).transform(texts)


def build_feature_matrix(texts, vectorizer, fit=True):
    """Combina TF-IDF com features estilísticas."""
    X_tfidf = vectorizer.fit_transform(texts) if fit else vectorizer.transform(texts)
    X_style = np.array([extract_stylistic_features(t) for t in texts])
    return np.hstack([X_tfidf, X_style])


# Construir features
texts_train = train_df['Text'].tolist()
y_train     = np.array([LABEL2IDX[l] for l in train_df['Label']])
texts_test  = test_df['Text'].tolist()

vectorizer = TFIDFVectorizer(max_features=1500)
X_train = build_feature_matrix(texts_train, vectorizer, fit=True)
X_test  = build_feature_matrix(texts_test,  vectorizer, fit=False)

# Normalização Z-score
mu    = X_train.mean(axis=0)
sigma = X_train.std(axis=0) + 1e-8
X_train_n = (X_train - mu) / sigma
X_test_n  = (X_test  - mu) / sigma

print(f'Shape X_train: {X_train.shape}')
print(f'Shape X_test:  {X_test.shape}')


Shape X_train: (125, 1513)
Shape X_test:  (150, 1513)


In [6]:
# Split treino / validação (80/20)
N_train = len(y_train)
perm    = np.random.permutation(N_train)
split   = int(0.8 * N_train)
tr_idx, val_idx = perm[:split], perm[split:]

X_tr,  y_tr  = X_train_n[tr_idx],  y_train[tr_idx]
X_val, y_val = X_train_n[val_idx], y_train[val_idx]

print(f'Treino:    {len(y_tr)} exemplos')
print(f'Validação: {len(y_val)} exemplos')


Treino:    100 exemplos
Validação: 25 exemplos


## 4. Implementação dos Modelos (numpy apenas)


In [7]:
# ── Funções de Activação e Loss ──────────────────────────────────────────────

def softmax(Z):
    """Softmax numericamente estável."""
    Z_s = Z - Z.max(axis=1, keepdims=True)
    E   = np.exp(Z_s)
    return E / E.sum(axis=1, keepdims=True)

def relu(Z):          return np.maximum(0.0, Z)
def relu_grad(Z):     return (Z > 0).astype(np.float32)

def cross_entropy_loss(Y_pred, y_true):
    """Cross-entropy para classificação multi-classe."""
    return -np.mean(np.log(Y_pred[np.arange(len(y_true)), y_true] + 1e-9))

def accuracy(y_pred, y_true):
    return np.mean(y_pred == y_true)

print('Funções auxiliares definidas.')


Funções auxiliares definidas.


In [8]:
class LogisticRegression:
    """
    Regressão Logística Multi-classe (One-vs-All implícito via softmax).
    Optimização: Gradient Descent com regularização L2.
    """
    def __init__(self, n_features, n_classes, lr=0.05, lambda_reg=0.01, n_epochs=600):
        self.W         = np.random.randn(n_features, n_classes).astype(np.float32) * 0.01
        self.b         = np.zeros(n_classes, dtype=np.float32)
        self.lr        = lr
        self.lambda_reg = lambda_reg
        self.n_epochs  = n_epochs

    def predict_proba(self, X):
        return softmax(X @ self.W + self.b)

    def predict(self, X):
        return np.argmax(self.predict_proba(X), axis=1)

    def fit(self, X, y, verbose=True):
        N = len(y)
        history = []
        for epoch in range(self.n_epochs):
            P  = self.predict_proba(X)
            loss = cross_entropy_loss(P, y)
            dZ = P.copy()
            dZ[np.arange(N), y] -= 1
            dZ /= N
            dW = X.T @ dZ + self.lambda_reg * self.W
            db = dZ.sum(axis=0)
            self.W -= self.lr * dW
            self.b -= self.lr * db
            history.append(loss)
            if verbose and (epoch + 1) % 100 == 0:
                acc = accuracy(self.predict(X), y)
                print(f'  Epoch {epoch+1:4d}/{self.n_epochs}  loss={loss:.4f}  acc={acc:.3f}')
        return history

print('LogisticRegression definida.')


LogisticRegression definida.


In [9]:
class DNN:
    """
    Deep Neural Network implementada do zero com numpy.
    Características:
      - Múltiplas camadas ocultas com activação ReLU
      - Camada de saída com Softmax
      - Inicialização He (óptima para ReLU)
      - Optimizer Adam (adaptativo)
      - Dropout (regularização)
      - Regularização L2 nos pesos
      - Early Stopping baseado na loss de validação
      - Mini-batch gradient descent
    """
    def __init__(self, layer_sizes, dropout_rate=0.3, lambda_reg=0.001,
                 lr=0.001, n_epochs=400, batch_size=16, patience=40):
        self.layer_sizes  = layer_sizes
        self.dropout_rate = dropout_rate
        self.lambda_reg   = lambda_reg
        self.lr           = lr
        self.n_epochs     = n_epochs
        self.batch_size   = batch_size
        self.patience     = patience

        # Inicialização He
        self.W = [
            np.random.randn(layer_sizes[i], layer_sizes[i+1]).astype(np.float32)
            * np.sqrt(2.0 / layer_sizes[i])
            for i in range(len(layer_sizes)-1)
        ]
        self.b = [np.zeros(layer_sizes[i+1], dtype=np.float32)
                  for i in range(len(layer_sizes)-1)]

        # Momentos Adam
        self.mW = [np.zeros_like(w) for w in self.W]
        self.vW = [np.zeros_like(w) for w in self.W]
        self.mb = [np.zeros_like(b) for b in self.b]
        self.vb = [np.zeros_like(b) for b in self.b]
        self.t  = 0
        self.best_W = self.best_b = None

    def _forward(self, X, training=False):
        self._acts  = [X]
        self._zs    = []
        self._masks = []
        A = X
        for i in range(len(self.W) - 1):
            Z = A @ self.W[i] + self.b[i]
            self._zs.append(Z)
            A = relu(Z)
            if training and self.dropout_rate > 0:
                mask = (np.random.rand(*A.shape) > self.dropout_rate).astype(np.float32)
                mask /= (1.0 - self.dropout_rate)
                A = A * mask
            else:
                mask = np.ones_like(A)
            self._masks.append(mask)
            self._acts.append(A)
        Z_out = A @ self.W[-1] + self.b[-1]
        self._zs.append(Z_out)
        A_out = softmax(Z_out)
        self._acts.append(A_out)
        return A_out

    def _backward(self, y):
        N  = len(y)
        nL = len(self.W)
        dA = self._acts[-1].copy()
        dA[np.arange(N), y] -= 1
        dA /= N
        gW = [None] * nL
        gb = [None] * nL
        for i in range(nL - 1, -1, -1):
            gW[i] = self._acts[i].T @ dA + self.lambda_reg * self.W[i]
            gb[i] = dA.sum(axis=0)
            if i > 0:
                dA = (dA @ self.W[i].T) * self._masks[i-1] * relu_grad(self._zs[i-1])
        return gW, gb

    def _adam(self, gW, gb, b1=0.9, b2=0.999, eps=1e-8):
        self.t += 1
        for i in range(len(self.W)):
            self.mW[i] = b1*self.mW[i] + (1-b1)*gW[i]
            self.vW[i] = b2*self.vW[i] + (1-b2)*gW[i]**2
            self.mb[i] = b1*self.mb[i] + (1-b1)*gb[i]
            self.vb[i] = b2*self.vb[i] + (1-b2)*gb[i]**2
            mW_ = self.mW[i] / (1-b1**self.t)
            vW_ = self.vW[i] / (1-b2**self.t)
            mb_ = self.mb[i] / (1-b1**self.t)
            vb_ = self.vb[i] / (1-b2**self.t)
            self.W[i] -= self.lr * mW_ / (np.sqrt(vW_) + eps)
            self.b[i] -= self.lr * mb_ / (np.sqrt(vb_) + eps)

    def fit(self, X_train, y_train, X_val=None, y_val=None, verbose=True):
        N = len(y_train)
        best_val = np.inf
        wait = 0
        hist = {'train_loss': [], 'val_loss': [], 'val_acc': []}

        for epoch in range(self.n_epochs):
            perm = np.random.permutation(N)
            Xs, ys = X_train[perm], y_train[perm]
            bloss = []
            for s in range(0, N, self.batch_size):
                Xb, yb = Xs[s:s+self.batch_size], ys[s:s+self.batch_size]
                Yp = self._forward(Xb, training=True)
                bloss.append(cross_entropy_loss(Yp, yb))
                gW, gb = self._backward(yb)
                self._adam(gW, gb)
            train_loss = np.mean(bloss)
            hist['train_loss'].append(train_loss)

            if X_val is not None:
                Yv = self._forward(X_val)
                vl = cross_entropy_loss(Yv, y_val)
                va = accuracy(np.argmax(Yv, axis=1), y_val)
                hist['val_loss'].append(vl); hist['val_acc'].append(va)
                if vl < best_val:
                    best_val = vl
                    self.best_W = [w.copy() for w in self.W]
                    self.best_b = [b.copy() for b in self.b]
                    wait = 0
                else:
                    wait += 1
                    if wait >= self.patience:
                        if verbose: print(f'  Early stopping @ epoch {epoch+1}')
                        break
                if verbose and (epoch+1) % 50 == 0:
                    print(f'  Epoch {epoch+1:4d}  train={train_loss:.4f}  val={vl:.4f}  acc={va:.3f}')
            else:
                if verbose and (epoch+1) % 50 == 0:
                    print(f'  Epoch {epoch+1:4d}  train={train_loss:.4f}')

        if self.best_W:
            self.W, self.b = self.best_W, self.best_b
        return hist

    def predict(self, X):
        return np.argmax(self._forward(X), axis=1)

    def predict_proba(self, X):
        return self._forward(X)

print('DNN definida.')


DNN definida.


## 5. Treino e Avaliação


In [10]:
n_feat    = X_train_n.shape[1]
n_classes = len(CLASSES)

print('── Regressão Logística (baseline) ──')
lr_model = LogisticRegression(n_feat, n_classes, lr=0.1, lambda_reg=0.01, n_epochs=500)
lr_model.fit(X_tr, y_tr, verbose=False)

lr_tr_acc  = accuracy(lr_model.predict(X_tr),  y_tr)
lr_val_acc = accuracy(lr_model.predict(X_val), y_val)
print(f'  Train acc: {lr_tr_acc:.3f}  |  Val acc: {lr_val_acc:.3f}')


── Regressão Logística (baseline) ──
  Train acc: 1.000  |  Val acc: 0.120


In [11]:
print('── DNN (numpy) ──')
dnn = DNN(
    layer_sizes  = [n_feat, 256, 128, 64, n_classes],
    dropout_rate = 0.4,
    lambda_reg   = 0.001,
    lr           = 0.001,
    n_epochs     = 500,
    batch_size   = 16,
    patience     = 40,
)
hist = dnn.fit(X_tr, y_tr, X_val=X_val, y_val=y_val, verbose=True)

dnn_tr_acc  = accuracy(dnn.predict(X_tr),  y_tr)
dnn_val_acc = accuracy(dnn.predict(X_val), y_val)
print(f'  Train acc: {dnn_tr_acc:.3f}  |  Val acc: {dnn_val_acc:.3f}')


── DNN (numpy) ──
  Epoch   50  train=0.0673  val=2.4019  acc=0.240
  Early stopping @ epoch 97
  Train acc: 1.000  |  Val acc: 0.320


In [12]:
print('── Comparação dos modelos na validação ──')
print(f'  Logistic Regression: {lr_val_acc:.3f}')
print(f'  DNN:                 {dnn_val_acc:.3f}')


── Comparação dos modelos na validação ──
  Logistic Regression: 0.120
  DNN:                 0.320


## 6. Re-treino no Conjunto Completo e Previsão da Submissão


In [13]:
print('── Re-treinar no conjunto completo para submissão ──')

lr_final = LogisticRegression(n_feat, n_classes, lr=0.1, lambda_reg=0.01, n_epochs=600)
lr_final.fit(X_train_n, y_train, verbose=False)

dnn_final = DNN(
    layer_sizes  = [n_feat, 256, 128, 64, n_classes],
    dropout_rate = 0.3,
    lambda_reg   = 0.001,
    lr           = 0.001,
    n_epochs     = 400,
    batch_size   = 16,
    patience     = 50,
)
dnn_final.fit(X_train_n, y_train, verbose=False)
print('Re-treino concluído.')


── Re-treinar no conjunto completo para submissão ──
Re-treino concluído.


In [14]:
# Ensemble: média ponderada das probabilidades
proba_lr  = lr_final.predict_proba(X_test_n)
proba_dnn = dnn_final.predict_proba(X_test_n)
proba_ens = 0.4 * proba_lr + 0.6 * proba_dnn
y_pred    = np.argmax(proba_ens, axis=1)
labels    = [IDX2LABEL[i] for i in y_pred]

print('Distribuição das previsões:')
print(Counter(labels))

# Guardar modelo
model_data = {
    'vectorizer': vectorizer, 'mu': mu, 'sigma': sigma,
    'lr_model': lr_final, 'dnn_model': dnn_final, 'classes': CLASSES
}
with open(MODEL_PATH, 'wb') as f:
    pickle.dump(model_data, f)
print(f'Modelo guardado: {MODEL_PATH}')

# Gerar CSV de submissão
out_df = test_df[['ID', 'Text']].copy()
out_df['Label'] = labels
out_df.to_csv(OUT_PATH, sep=';', index=False)
print(f'CSV gerado: {OUT_PATH}')
out_df.head(10)


Distribuição das previsões:
Counter({'Human': 57, 'Anthropic': 29, 'OpenAI': 28, 'Meta': 27, 'Google': 9})
Modelo guardado: model_A.pkl
CSV gerado: subm1-g7-MIA-A.csv


,ID,Text,Label
0,D2-1,A covalent bond is a chemical bond that involv...,Human
1,D2-2,A covalent bond forms when two atoms share one...,OpenAI
2,D2-3,A covalent bond is a type of chemical bond whe...,OpenAI
3,D2-4,A covalent bond is a chemical bond that involv...,Meta
4,D2-5,Driven by exciting developments in the field o...,Anthropic
5,D2-6,Ionic bonding results from the electrostatic a...,Human
6,D2-7,Plate tectonics is the scientific theory expla...,Anthropic
7,D2-8,Plate tectonics is a fundamental scientific th...,OpenAI
8,D2-9,Tectonic plates are relatively rigid and float...,OpenAI
9,D2-10,"At present, young oceanic lithosphere is posit...",Meta
